In [1]:
from pathlib import Path

import numpy as np
import scipy
import xarray as xr

In [2]:
from data import ds_pressure, ds_single

In [3]:
def compute_bit_information(
    x: np.ndarray[tuple[int, ...], np.dtype[np.float32]],
    set_zero_insignificant_confidence: None | float = None,
) -> np.ndarray[tuple[32], np.dtype[np.float64]]:
    assert x.dtype.type is np.float32

    x = x.ravel().view(np.uint32)

    if len(x) == 0:
        # no data elements -> every bit has zero information
        return np.zeros(32)

    if len(x) == 1:
        # only one element -> every bit has full information
        return np.ones(32)

    front, back = x[:-1], x[1:]
    n_elements = len(front)

    bitpair_counter = np.zeros((32, 2, 2), dtype=np.uintp)
    for bi in range(32):
        j = ((front >> bi) & 1) == 1
        k = ((back >> bi) & 1) == 1

        bitpair_counter[32 - bi - 1, 0, 0] += np.sum((~j) & (~k))
        bitpair_counter[32 - bi - 1, 0, 1] += np.sum((~j) & k)
        bitpair_counter[32 - bi - 1, 1, 0] += np.sum(j & (~k))
        bitpair_counter[32 - bi - 1, 1, 1] += np.sum(j & k)

    joint_probability_mass_function = bitpair_counter / n_elements

    mutual_information = np.zeros(32)
    p = joint_probability_mass_function
    # marginal probabilities of y
    py = np.stack([p[:, 0, 0] + p[:, 1, 0], p[:, 0, 1] + p[:, 1, 1]])
    # marginal probabilities of x
    px = np.stack([p[:, 0, 0] + p[:, 0, 1], p[:, 1, 0] + p[:, 1, 1]])
    with np.errstate(divide="ignore", over="ignore", under="ignore", invalid="ignore"):
        for i in range(2):
            for j in range(2):
                mutual_information += np.where(
                    p[:, i, j] > 0.0,
                    p[:, i, j] * np.log2(p[:, i, j] / px[i] / py[j]),
                    0.0,
                )

    if set_zero_insignificant_confidence is not None:
        free_entropy_of_50_50 = binom_free_binary_entropy(
            n_elements, set_zero_insignificant_confidence
        )
        mutual_information[mutual_information <= free_entropy_of_50_50] = 0.0

    return mutual_information


def binom_free_binary_entropy(n: int, confidence: float) -> float:
    p = binom_confidence(n, confidence)

    binary_entropy = -((p * np.log2(p)) + ((1.0 - p) * np.log2(1.0 - p)))

    return 1.0 - binary_entropy


def binom_confidence(n: int, confidence: float) -> float:
    p = 0.5 + standard_normal_quantile(1.0 - ((1.0 - confidence) * 0.5)) / (
        np.sqrt(n) * 2.0
    )

    # cap probability at 1 (only important for when n is small)
    return min(p, 1.0)


def standard_normal_quantile(p: float) -> float:
    return scipy.stats.norm.ppf(p)

In [4]:
def compute_clean_bitinformation(xs: np.array) -> tuple[float, ...]:
    bit_information = compute_bit_information(
        xs, set_zero_insignificant_confidence=0.99
    )

    # find the first per-bit information maximum in the mantissa
    # round to two decimal places to improve stability
    bit_information_mantissa = bit_information[9:]
    increasing = np.round(bit_information_mantissa[1:], 2) >= np.round(
        bit_information_mantissa[:-1], 2
    )
    imax = np.where(~increasing)[0]
    if len(imax) == 0:
        # no false information at the end
        return tuple(bit_information)
    imax = imax[0]

    # find the first per-bit information maximum in the mantissa
    # round to two decimal places to improve stability
    bit_information_skyfall = bit_information[9 + imax :]
    decreasing = np.round(bit_information_skyfall[1:], 2) <= np.round(
        bit_information_skyfall[:-1], 2
    )
    imin = np.where(~decreasing)[0]
    if len(imin) == 0:
        # no false information at the end
        return tuple(bit_information)
    imin = imin[0]

    bit_information_corrected = np.copy(bit_information)
    bit_information_corrected[9 + imax + imin + 1 :] = 0.0

    return tuple(bit_information_corrected)

In [5]:
from compression_recommendations import Recommendations
from compression_recommendations.filters.abc import Filter
from compression_recommendations.filters.cf import (
    CfShortNameFilter,
    CfStandardNameFilter,
)
from compression_recommendations.filters.combinators import AnyFilter
from compression_recommendations.filters.grib import GribShortNameFilter
from compression_recommendations.filters.level import (
    LevelKind,
    LevelKindFilter,
    LevelValueFilter,
)
from compression_recommendations.filters.tag import TagFilter
from compression_recommendations.recommendation import Recommendation
from compression_recommendations.requirements.error_bounds.max import (
    MaxPointwiseRelativeErrorBoundRequirement,
)
from semver import Version

In [6]:
def compute_bitinformation_recommendations(
    da: xr.DataArray,
    level: LevelKind,
    percentiles: list[float],
    extra_filters: list[Filter],
) -> list[Recommendation]:
    xs = da.values.flatten()

    bit_information = compute_clean_bitinformation(xs)

    base_filters = [
        AnyFilter(
            filters=(
                [CfStandardNameFilter(value=da.standard_name)]
                if da.standard_name.lower() != "unknown"
                else []
            )
            + [
                CfShortNameFilter(value=da.name),
                GribShortNameFilter(value=da.GRIB_shortName),
            ]
        ),
        LevelKindFilter(value=level),
    ]

    recommendations = []

    for p in percentiles:
        filters = base_filters + [TagFilter(value=f"{p * 100}%")] + extra_filters

        # constant variable
        if np.sum(bit_information) == 0.0:
            recommendations.append(
                Recommendation(
                    filters=filters,
                    requirements=[MaxPointwiseRelativeErrorBoundRequirement(value=0)],
                )
            )
            continue

        keepbits = (
            np.searchsorted(np.cumsum(bit_information) / np.sum(bit_information), p) + 1
        )
        keepbits_mantissa = keepbits - 9

        # Without the mantissa, we can still represent every 2^k.
        # The maximum distance between consecutive 2^(k-1), 2^k, 2^(k+1) is 2^k,
        #  or 1 relative to 2^k.
        # If round-to-even is used when removing bits, the relative error will
        #  be half of that in the worst case.
        eb_rel = np.exp2(-keepbits_mantissa) * 0.5

        recommendations.append(
            Recommendation(
                filters=filters,
                requirements=[MaxPointwiseRelativeErrorBoundRequirement(value=eb_rel)],
            )
        )

    return recommendations

In [7]:
from tqdm import tqdm

In [8]:
recommendations = []

In [9]:
for key in tqdm(sorted(ds_single.mean)):
    recommendations += compute_bitinformation_recommendations(
        da=ds_single.mean[key],
        level=LevelKind.single,
        percentiles=[1.0, 0.999, 0.99, 0.95],
        extra_filters=[],
    )

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 226/226 [33:10<00:00,  8.81s/it]


In [10]:
for key in tqdm(sorted(ds_pressure.mean)):
    for pressure in sorted(ds_pressure.mean[key].pressure_level.values):
        recommendations += compute_bitinformation_recommendations(
            da=ds_pressure.mean[key].sel(pressure_level=pressure),
            level=LevelKind.pressure,
            percentiles=[1.0, 0.999, 0.99, 0.95],
            extra_filters=[LevelValueFilter(minimum=pressure, maximum=pressure)],
        )

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [09:04<00:00, 34.05s/it]


In [11]:
with Path("recommendations").joinpath("bit-information.yaml").open("w") as f:
    Recommendations(
        recommendations=recommendations,
        version=Version.parse("0.1.0"),
        metadata={"commit": ""},
    ).dump(f)